# Optimized NQS ground-state notebook (Cambridge HPC / CSD3)

Uses `NQS_hpc.py`: same RBM / TFI / VMC / SR physics as the legacy `NQS.py`, with a
**batched multi-chain MCMC sampler** and device-agnostic tensors, so this actually benefits
from a GPU. Open this notebook through CSD3's Jupyter web interface, requesting a GPU (Ampere
partition) session, to run the scale-up cells below. The **Setup** section right below
checks/installs what's needed directly from here -- no separate SSH session or manual
`ipykernel install` step required; it runs inside whatever Jupyter environment CSD3 gives
you for that session.

The small-`N` correctness check and MCMC-vs-ED benchmark are identical to
`spin_chain_ground_optimized.ipynb` (the non-HPC version) — kept here too as a sanity check
before trusting the larger runs further down. `NQS.py` is used (fixed) for the Legacy section of
`spin_chain_ground_optimized.ipynb`, which the standalone `spin_chain_ground.ipynb` has been
folded into and discarded from. One correctness fix was applied to both `NQS.py` and here: the
Hamiltonian's local-energy formula had a sign/scaling bug (see the non-HPC notebook's intro
cell for the full explanation) that has been fixed in `NQS.py`, `NQS_hpc.py`, and here.

## Setup

Run this once per session. It checks the kernel CSD3's Jupyter environment is already using, and only installs a package if it's actually missing -- it won't blindly reinstall `torch` over a CUDA build CSD3 may have already matched to this node's driver.

In [ ]:
import importlib
import subprocess
import sys


def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pkg} into this kernel's environment...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)


for pkg in ("numpy", "scipy", "pandas", "matplotlib"):
    ensure(pkg)

ensure("torch")
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print()
    print("No CUDA-enabled GPU visible in this kernel.")
    print("- If this session wasn't launched on CSD3's GPU (Ampere) partition, relaunch there.")
    print("- If it IS a GPU session but torch still isn't finding CUDA, this kernel's torch")
    print("  build is probably CPU-only. Check `nvidia-smi` below for the driver/CUDA version")
    print("  this node exposes, then reinstall torch matching it, e.g.:")
    print('    %pip install -q torch --index-url https://download.pytorch.org/whl/cu121')


In [ ]:
!nvidia-smi

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from NQS_hpc import ed_ground, run_training, RBM, TFI, BatchedMCMC

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


## Correctness check

Same `N=4, M=8, J=1, h=2` setup as the legacy notebook. The VMC energy should converge to
the exact ground energy from `ed_ground`.

In [ ]:
rbm, energy_history = run_training(
    N=4, M=8, J=1.0, h=2.0, seed=0,
    n_chains=64, n_steps=50, burn_in=100, n_iterations=80,
    device=DEVICE,
)

E_exact = ed_ground(4, 1.0, 2.0)
E_vmc = energy_history[-1]

print(f"exact    E_0 = {E_exact:.6f}")
print(f"VMC      <E> = {E_vmc:.6f}")
print(f"error        = {E_vmc - E_exact:.6f}  ({100 * abs(E_vmc - E_exact) / abs(E_exact):.2f} %)")

fig = plt.figure()
plt.plot(energy_history, marker='o', linestyle='-', label='RBM + SR (batched)')
plt.axhline(E_exact, color='k', linestyle='--', label='exact (ED)')
plt.xlabel('Iteration')
plt.ylabel('Energy')
plt.title(f'N={rbm.N}, M={rbm.M}, h/J={2.0/1.0:g}, device={DEVICE}')
plt.legend()
plt.show()


## MCMC vs ED benchmark

Same small-`N` validation as the non-HPC notebook: VMC energy vs exact (ED) energy, across
two sweeps. Exact diagonalization is a small-`N` benchmark only — it is **not** used in the
large-`N` scale-up section below.

### Energy vs `h/J`, for several fixed `N`

In [ ]:
def train_final_energy(N, M, J, h, seed, n_iterations=40, n_chains=64, n_steps=40, burn_in=40, solver="dense"):
    _, hist = run_training(
        N=N, M=M, J=J, h=h, seed=seed,
        n_chains=n_chains, n_steps=n_steps, burn_in=burn_in,
        n_iterations=n_iterations, device=DEVICE, solver=solver,
    )
    return float(np.mean(hist[-5:]))


Ns_for_ratio_sweep = [4, 6, 8, 10]
ratios = np.linspace(0.1, 4.0, 30)
J_fixed = 1.0

ratio_results = {N: {"mcmc": [], "ed": []} for N in Ns_for_ratio_sweep}

for N in Ns_for_ratio_sweep:
    M = 2 * N
    for ratio in ratios:
        h = ratio * J_fixed
        ratio_results[N]["mcmc"].append(train_final_energy(N, M, J_fixed, h, seed=0))
        ratio_results[N]["ed"].append(ed_ground(N, J_fixed, h))
    print(f"N={N} done")


In [ ]:
fig, axes = plt.subplots(1, len(Ns_for_ratio_sweep), figsize=(4 * len(Ns_for_ratio_sweep), 4), sharex=True)

for ax, N in zip(axes, Ns_for_ratio_sweep):
    ax.plot(ratios, ratio_results[N]["mcmc"], label="MCMC", marker='o', markersize=3, linestyle='-')
    ax.plot(ratios, ratio_results[N]["ed"], label="ED (exact)", linestyle='--')
    ax.set_title(f"N={N}")
    ax.set_xlabel("h / J")

axes[0].set_ylabel("Ground-state energy")
axes[0].legend()
fig.suptitle("MCMC vs ED: energy vs h/J, by N")
plt.tight_layout()
plt.show()


Note: at N=6/8 the MCMC curve sits visibly above ED for small h/J (the strongly-coupled, hardest-to-optimize regime) — that’s the light, fixed iteration/chain budget used here to keep the sweep fast (same settings regardless of N), not a bug: the correctness-check cell above, run with a larger budget, matches ED to 0.04%. Increase `n_iterations`/`n_chains` in `train_final_energy` (cheap on a GPU) to tighten agreement at larger N.

### Energy vs `N`, at fixed `h/J`

In [ ]:
Ns_for_N_sweep = list(range(2, 13))
h_over_J_fixed = 1.0

n_sweep_mcmc = []
n_sweep_ed = []

for N in Ns_for_N_sweep:
    M = 2 * N
    h = h_over_J_fixed * J_fixed
    n_sweep_mcmc.append(train_final_energy(N, M, J_fixed, h, seed=0) / N)
    n_sweep_ed.append(ed_ground(N, J_fixed, h) / N)

fig = plt.figure()
plt.plot(Ns_for_N_sweep, n_sweep_mcmc, marker='o', linestyle='-', label='MCMC')
plt.plot(Ns_for_N_sweep, n_sweep_ed, marker='s', linestyle='--', label='ED (exact)')
plt.xlabel('N')
plt.ylabel('Energy density E / N')
plt.title(f'MCMC vs ED: energy density vs N, h/J={h_over_J_fixed:g}')
plt.legend()
plt.show()


## CPU vs GPU timing benchmark

Compares the **legacy** single-chain, one-spin-flip-at-a-time sampler against the
**batched** multi-chain sampler, and (when a GPU is present) CPU vs GPU for the batched
sampler. This isolates the actual speedup from batching/device placement, independent of
training dynamics.

In [ ]:
from NQS import RBM as LegacyRBM, MCMC as LegacyMCMC

def time_legacy_sampler(N, M, n_samples, burn_in=50, seed=0):
    rbm = LegacyRBM(num_visible=N, num_hidden=M, seed=seed)
    mcmc = LegacyMCMC(n_spins=N, seed=seed)
    t0 = time.time()
    mcmc.sample(rbm, n_samples=n_samples, burn_in=burn_in)
    return time.time() - t0


def time_batched_sampler(N, M, n_chains, n_steps, device, burn_in=50, seed=0):
    rbm = RBM(num_visible=N, num_hidden=M, device=device, seed=seed)
    mcmc = BatchedMCMC(n_spins=N, n_chains=n_chains, device=device, seed=seed)
    t0 = time.time()
    mcmc.sample(rbm, n_steps=n_steps, burn_in=burn_in)
    if device == "cuda":
        torch.cuda.synchronize()
    return time.time() - t0


N_bench, M_bench = 16, 32
n_total_samples = 2000  # legacy: 2000 sequential single-chain samples

t_legacy = time_legacy_sampler(N_bench, M_bench, n_samples=n_total_samples)
t_batched_cpu = time_batched_sampler(N_bench, M_bench, n_chains=64, n_steps=n_total_samples // 64, device="cpu")

print(f"Legacy single-chain sampler   ({n_total_samples} samples, CPU):  {t_legacy:.3f} s")
print(f"Batched sampler (64 chains, CPU):                  {t_batched_cpu:.3f} s  "
      f"({t_legacy / t_batched_cpu:.1f}x speedup from batching alone)")

if torch.cuda.is_available():
    t_batched_gpu = time_batched_sampler(N_bench, M_bench, n_chains=64, n_steps=n_total_samples // 64, device="cuda")
    print(f"Batched sampler (64 chains, GPU):                  {t_batched_gpu:.3f} s  "
          f"({t_legacy / t_batched_gpu:.1f}x speedup over legacy)")
else:
    print("No CUDA device visible — run this cell on a CSD3 GPU (Ampere partition) node/kernel "
          "to see the GPU numbers.")


## Scale-up runs

Train at larger `N` with more hidden units and chains — intended for a CSD3 GPU session.
No exact-diagonalization check here (benchmark-only, see above); `solver="cg"` avoids
forming the full `(N+M+N*M)²` Fisher matrix, which otherwise dominates cost at these sizes.

**Scaling knobs**: once running on an actual GPU, increase `n_chains` (more parallel chains
is close to free on GPU until memory-bound) and `n_iterations` well beyond the CPU-sized
defaults below — the numbers here are deliberately modest so this cell is still a reasonable
smoke test if run on CPU.

In [ ]:
scale_up_configs = [
    dict(N=16, M=32, n_chains=128, n_steps=80, burn_in=80, n_iterations=100),
    dict(N=20, M=40, n_chains=128, n_steps=80, burn_in=80, n_iterations=100),
    dict(N=24, M=48, n_chains=128, n_steps=80, burn_in=80, n_iterations=100),
]

scale_up_results = {}
for cfg in scale_up_configs:
    N = cfg["N"]
    t0 = time.time()
    rbm, hist = run_training(J=1.0, h=1.0, seed=0, device=DEVICE, solver="cg", **cfg)
    dt = time.time() - t0
    scale_up_results[N] = hist
    print(f"N={N}: final energy = {hist[-1]:.4f}, energy/N = {hist[-1] / N:.4f}, "
          f"wall time = {dt:.1f} s ({dt / cfg['n_iterations']:.3f} s/iteration)")


In [ ]:
fig = plt.figure()
for N, hist in scale_up_results.items():
    plt.plot(hist, label=f"N={N}")
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.title(f"Scale-up training curves (device={DEVICE})")
plt.legend()
plt.show()


## Seed sweep and `h/J` sweep at larger scale

Same finer-grid sweep as the non-HPC notebook, re-run at a larger `N`/more chains — cheap
to push much further here given GPU batching.

In [ ]:
def minimise_e(seed, N, M, n_iterations=50, n_samples_steps=50, burn_in=50,
               J=1.0, h=2.0, noise_std=0.0, solver="dense"):
    _, hist = run_training(
        N=N, M=M, J=J, h=h, seed=seed,
        n_chains=128, n_steps=n_samples_steps, burn_in=burn_in,
        n_iterations=n_iterations, device=DEVICE, noise_std=noise_std, solver=solver,
    )
    return hist


N_sweep_scale, M_sweep_scale = 16, 32

n_seeds = 50
seed_energies = [
    minimise_e(seed, N_sweep_scale, M_sweep_scale, solver="cg")[-1] for seed in range(n_seeds)
]

seed_energies_arr = np.array(seed_energies)
stats_table = pd.DataFrame({
    "Statistic": ["Mean", "Min", "Max", "Variance"],
    "Energy (final iteration)": [
        seed_energies_arr.mean(),
        seed_energies_arr.min(),
        seed_energies_arr.max(),
        seed_energies_arr.var(),
    ],
})
stats_table


In [ ]:
ratios_fine = np.linspace(0.1, 10.0, 60)
n_seeds_sweep = 20

min_energy_by_ratio = []
for ratio in ratios_fine:
    h = ratio * 1.0
    energies = [
        minimise_e(seed, N_sweep_scale, M_sweep_scale, n_iterations=40, n_samples_steps=50,
                   burn_in=40, h=h, solver="cg")[-1]
        for seed in range(n_seeds_sweep)
    ]
    min_energy_by_ratio.append(np.min(energies))

fig = plt.figure()
plt.plot(ratios_fine, min_energy_by_ratio, marker='o', markersize=3, linestyle='-')
plt.xlabel('h / J')
plt.ylabel(f'Minimum energy across {n_seeds_sweep} seeds')
plt.title(f'N={N_sweep_scale}, M={M_sweep_scale}: fine h/J sweep ({len(ratios_fine)} points)')
plt.show()
